In [2]:
import sys
import json
from types import SimpleNamespace

import numpy as np
import torch
from sklearn.metrics import roc_curve, roc_auc_score

sys.path.append("../")
from clip import clip
from src.utils.dataset import CUBWithFilename

In [3]:
classnames_cub = ['black footed albatross', 'laysan albatross', 'sooty albatross', 'groove billed ani', 'crested auklet', 'least auklet', 'parakeet auklet', 'rhinoceros auklet', 'brewer blackbird', 'red winged blackbird', 'rusty blackbird', 'yellow headed blackbird', 'bobolink', 'indigo bunting', 'lazuli bunting', 'painted bunting', 'cardinal', 'spotted catbird', 'gray catbird', 'yellow breasted chat', 'eastern towhee', 'chuck will widow', 'brandt cormorant', 'red faced cormorant', 'pelagic cormorant', 'bronzed cowbird', 'shiny cowbird', 'brown creeper', 'american crow', 'fish crow', 'black billed cuckoo', 'mangrove cuckoo', 'yellow billed cuckoo', 'gray crowned rosy finch', 'purple finch', 'northern flicker', 'acadian flycatcher', 'great crested flycatcher', 'least flycatcher', 'olive sided flycatcher', 'scissor tailed flycatcher', 'vermilion flycatcher', 'yellow bellied flycatcher', 'frigatebird', 'northern fulmar', 'gadwall', 'american goldfinch', 'european goldfinch', 'boat tailed grackle', 'eared grebe', 'horned grebe', 'pied billed grebe', 'western grebe', 'blue grosbeak', 'evening grosbeak', 'pine grosbeak', 'rose breasted grosbeak', 'pigeon guillemot', 'california gull', 'glaucous winged gull', 'heermann gull', 'herring gull', 'ivory gull', 'ring billed gull', 'slaty backed gull', 'western gull', 'anna hummingbird', 'ruby throated hummingbird', 'rufous hummingbird', 'green violetear', 'long tailed jaeger', 'pomarine jaeger', 'blue jay', 'florida jay', 'green jay', 'dark eyed junco', 'tropical kingbird', 'gray kingbird', 'belted kingfisher', 'green kingfisher', 'pied kingfisher', 'ringed kingfisher', 'white breasted kingfisher', 'red legged kittiwake', 'horned lark', 'pacific loon', 'mallard', 'western meadowlark', 'hooded merganser', 'red breasted merganser', 'mockingbird', 'nighthawk', 'clark nutcracker', 'white breasted nuthatch', 'baltimore oriole', 'hooded oriole', 'orchard oriole', 'scott oriole', 'ovenbird', 'brown pelican', 'white pelican', 'western wood pewee', 'sayornis', 'american pipit', 'whip poor will', 'horned puffin', 'common raven', 'white necked raven', 'american redstart', 'geococcyx', 'loggerhead shrike', 'great grey shrike', 'baird sparrow', 'black throated sparrow', 'brewer sparrow', 'chipping sparrow', 'clay colored sparrow', 'house sparrow', 'field sparrow', 'fox sparrow', 'grasshopper sparrow', 'harris sparrow', 'henslow sparrow', 'le conte sparrow', 'lincoln sparrow', 'nelson sharp tailed sparrow', 'savannah sparrow', 'seaside sparrow', 'song sparrow', 'tree sparrow', 'vesper sparrow', 'white crowned sparrow', 'white throated sparrow', 'cape glossy starling', 'bank swallow', 'barn swallow', 'cliff swallow', 'tree swallow', 'scarlet tanager', 'summer tanager', 'artic tern', 'black tern', 'caspian tern', 'common tern', 'elegant tern', 'forsters tern', 'least tern', 'green tailed towhee', 'brown thrasher', 'sage thrasher', 'black capped vireo', 'blue headed vireo', 'philadelphia vireo', 'red eyed vireo', 'warbling vireo', 'white eyed vireo', 'yellow throated vireo', 'bay breasted warbler', 'black and white warbler', 'black throated blue warbler', 'blue winged warbler', 'canada warbler', 'cape may warbler', 'cerulean warbler', 'chestnut sided warbler', 'golden winged warbler', 'hooded warbler', 'kentucky warbler', 'magnolia warbler', 'mourning warbler', 'myrtle warbler', 'nashville warbler', 'orange crowned warbler', 'palm warbler', 'pine warbler', 'prairie warbler', 'prothonotary warbler', 'swainson warbler', 'tennessee warbler', 'wilson warbler', 'worm eating warbler', 'yellow warbler', 'northern waterthrush', 'louisiana waterthrush', 'bohemian waxwing', 'cedar waxwing', 'american three toed woodpecker', 'pileated woodpecker', 'red bellied woodpecker', 'red cockaded woodpecker', 'red headed woodpecker', 'downy woodpecker', 'bewick wren', 'cactus wren', 'carolina wren', 'house wren', 'marsh wren', 'rock wren', 'winter wren', 'common yellowthroat']

In [4]:
args = SimpleNamespace(
    image_dir = '/home/user/Develop/datasets/cub/CUB_200_2011/images/test'
)
args.checkpoint_path = '../checkpoints_acc4/cub_visible_all_vit16/lambda_2b/ours_lambda6.0_ep1_v4/run_/best.pt'

In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/16", device=device, jit=False)
checkpoint = torch.load(args.checkpoint_path)
model.load_state_dict(checkpoint['model_state_dict'])
print(f'load... {args.checkpoint_path}')

load... ../checkpoints_acc4/cub_visible_all_vit16/lambda_2b/ours_lambda6.0_ep1_v4/run_/best.pt


/tmp/ipykernel_25687/1209090854.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(args.checkpoint_path)


In [6]:
dataset = CUBWithFilename(args.image_dir, transform=preprocess, mode='all')

In [7]:
with open("../descriptors/my_cub2-1.json", "r") as f:
    descriptors = json.load(f)

In [8]:
concept_ids_train = []
for classname in classnames_cub:
    class_concept = descriptors[classname]
    join_concept = ", ".join(class_concept[:-1]) + " and " + class_concept[-1]
    class_concept = [f"{classname} with {x}." for x in class_concept]
    class_concept = [join_concept] + class_concept
    concept_ids_train.append(clip.tokenize(class_concept).unsqueeze(0))
concept_ids_train = torch.concat(concept_ids_train)
text_ids = concept_ids_train

In [9]:
model.eval()
all_text_features = []
for i in range(text_ids.size(0)):
    text_features = model.encode_text(text_ids[i].to(device))
    text_features = text_features.detach().cpu()
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)
    all_text_features.append(text_features.unsqueeze(0))
text_features = torch.concat(all_text_features)
text_features.shape

torch.Size([200, 11, 512])

In [10]:
with open('../cub_visible_part_bench/anno_cub_test.json', 'r') as f:
    anno = json.load(f)

print(f'annotations: {len(anno)} images')
sample_key = list(anno.keys())[0]
print('sample:', sample_key, '->', anno[sample_key])

annotations: 5794 images
sample: Black_Footed_Albatross_0046_18.jpg -> {'head': 1, 'eye': 1, 'beak': 1, 'neck': 1, 'breast': 0, 'belly (underparts)': 0, 'back': 0, 'wing': 0, 'leg': 0, 'tail': 0}


In [11]:
def compute_fpr95(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores)
    if np.max(tpr) < 0.95:
        return np.nan
    idx = np.where(tpr >= 0.95)[0][0]
    return fpr[idx]


def compute_imagewise_auroc_fpr95(dataset, model, text_features, anno, device):
    model.eval()
    aurocs = []
    fpr95s = []
    scores_all = []

    with torch.no_grad():
        for sample in dataset:
            pixel_values = sample[0][None, :].to(device)
            class_id = sample[1]
            filename = sample[2]

            if filename not in anno:
                continue

            visible_mask = list(anno[filename].values())
            visible_mask = torch.tensor(visible_mask)

            image_features = model.encode_image(pixel_values)
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)
            scores = (
                image_features.cpu()
                @ text_features[class_id, 1:].T
            ).numpy()[0]

            labels = visible_mask.cpu().numpy()
            mask = (labels != 0.5)
            labels_bin = labels[mask].astype(np.int32)
            scores = scores[mask]

            if len(np.unique(labels_bin)) < 2:
                continue

            scores_all.append((scores, labels_bin))
            aurocs.append(roc_auc_score(labels_bin, scores))

            fpr95 = compute_fpr95(labels_bin, scores)
            if not np.isnan(fpr95):
                fpr95s.append(fpr95)

    print(f'evaluated images: {len(aurocs)}')
    print(f'AUROC: {float(np.mean(aurocs)):.4f}')
    print(f'FPR95: {float(np.mean(fpr95s)):.4f}')
    return scores_all

In [12]:
print(args.checkpoint_path)
scores_all = compute_imagewise_auroc_fpr95(dataset, model, text_features, anno, device)

../checkpoints_acc4/cub_visible_all_vit16/lambda_2b/ours_lambda6.0_ep1_v4/run_/best.pt
evaluated images: 2643
AUROC: 0.6116
FPR95: 0.7800
